In [2]:
!pip install peft -q

import json
import random
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ── Paths ────────────────────────────────────────────────────
BASE_MODEL_PATH = os.path.join(
    os.path.expanduser("~"),
    "Desktop", "Qwen_CustomDomain", "Model", "Qwen2.5-3B-Instruct"
)

CHECKPOINT_PATH = os.path.join(
    os.path.expanduser("~"),
    "Desktop", "Qwen_CustomDomain", "Checkpoints", "checkpoint-1782"
)

DATA_PATH = os.path.join(
    os.path.expanduser("~"),
    "Desktop", "Qwen_CustomDomain", "Dataset", "qa_pairs_100.jsonl"
)

# ── Clean GPU Memory ─────────────────────────────────────────
import gc
gc.collect()
torch.cuda.empty_cache()

# ── 4-bit config ─────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("[1/3] Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_PATH,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("[2/3] Loading base model in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True
)

print("[3/3] Loading LoRA adapter from checkpoint-1782...")
model = PeftModel.from_pretrained(base_model, CHECKPOINT_PATH)
model.eval()

print("✅ Model ready\n")

# ── Load 10 questions ────────────────────────────────────────
with open(DATA_PATH, "r", encoding="utf-8") as f:
    all_data = [json.loads(line) for line in f if line.strip()]

samples = random.sample(all_data, 10)

# ── Inference ────────────────────────────────────────────────
def ask_model(question):
    prompt = (
        f"<|im_start|>system\n"
        f"You are a Java expert assistant. "
        f"Answer only from your Java training knowledge. "
        f"If the question cannot be answered from your knowledge base, say: "
        f"I don't have enough information in my knowledge base to answer this."
        f"<|im_end|>\n"
        f"<|im_start|>user\n{question}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]

    return tokenizer.decode(
        outputs[0][prompt_len:],
        skip_special_tokens=True
    ).strip()

# ── Run 10 Questions ─────────────────────────────────────────
print("=" * 60)
print("   EPOCH 2 — OUTPUT QUALITY TEST (10 questions)")
print("=" * 60)

good = partial = poor = 0

for i, item in enumerate(samples):
    question = item["messages"][1]["content"].split("Question:")[-1].strip()
    expected = item["messages"][2]["content"]

    model_answer = ask_model(question)

    expected_words = set(expected[:150].lower().split())
    model_words = set(model_answer.lower().split())

    overlap = len(expected_words & model_words) / max(len(expected_words), 1)

    if overlap > 0.50:
        label = "✅ Good"
        good += 1
    elif overlap > 0.25:
        label = "🔍 Partial"
        partial += 1
    else:
        label = "❌ Poor"
        poor += 1

    print(f"\n[{i+1:02d}] Q        : {question[:120]}")
    print(f"      Expected : {expected[:200]}")
    print(f"      Model    : {model_answer[:200]}")
    print(f"      Match    : {label} ({overlap*100:.0f}% keyword overlap)")
    print("-" * 60)

# ── Summary ──────────────────────────────────────────────────
print("\n" + "=" * 60)

score = (good * 1.0 + partial * 0.5) / 10 * 100

print(f"Good     : {good}/10")
print(f"Partial  : {partial}/10")
print(f"Poor     : {poor}/10")
print(f"Score    : {score:.0f}%")

if score >= 70:
    print("Verdict  : ✅ Epoch 2 looks production-ready")
elif score >= 50:
    print("Verdict  : ⚠️ Usable — Epoch 3 may still improve recall")
else:
    print("Verdict  : ❌ More training likely needed")

print("=" * 60)


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


[1/3] Loading tokenizer...
[2/3] Loading base model in 4-bit...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[3/3] Loading LoRA adapter from checkpoint-1782...
✅ Model ready

   EPOCH 2 — OUTPUT QUALITY TEST (10 questions)

[01] Q        : Why are process and state fundamental ideas in proving program correctness?
      Expected : That is not explicitly stated in the text provided. However, it can be inferred that understanding states and processes helps in analyzing how a program behaves at different points during its executio
      Model    : Process and state are fundamental ideas in proving program correctness because they help break down complex systems into manageable components that can be analyzed individually. By understanding the c
      Match    : ✅ Good (57% keyword overlap)
------------------------------------------------------------

[02] Q        : What color should be used for the border around the selected square?
      Expected : I don't have enough information in my knowledge base to answer this.
      Model    : The color of the border around the selected square can vary d